In [ ]:
import torch
import torch.nn as nn

#Experimentation with classes, ANFIS layers, operations,...



##0 Parameters
These classes were made as an attemp to fix a problem with gradients calculation.

In [ ]:
class Premises(nn.Module):
    '''
    Class that represents the premises parameters of an ANFIS

    To initialize it:
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly
    '''
    def __init__(self, x_train=[]):
        super(Premises, self).__init__()
        if x_train == []:
            self.premises = torch.rand(input_size, rules, len(params))
        else:
            self.premises = torch.rand(input_size, rules, len(params))
            if self.rules > 1:
                min = torch.min(x_train, dim=0).values
                max = torch.max(x_train, dim=0).values
                stp = (max - min) / (self.rules - 1)
                for i in range(self.input_size):
                    h = torch.arange(min[i], max[i] + stp[i], stp[i])
                    for j in range(self.rules):
                        self.premises[i][j][0] = h[j]
                        self.premises[i][j][1] = stp[i]/2
                        if (len(self.params) == 3): self.premises[i][j][2] = 2
            else:
                for i in range(self.input_size):
                    for j in range(self.rules):
                        self.premises[i][j][0] = torch.std(x_train[:, i])
                        self.premises[i][j][1] = torch.mean(x_train[:, i])
                        if (len(self.params) == 3): self.premises[i][j][2] = 2
        self.premises.requires_grad = True

In [ ]:
class Consequents(nn.Module):
    '''
    Class that represents the consequents parameters of an ANFIS
    '''
    def __init__(self, x_train=[]):
        super(Consequents, self).__init__()
        self.premises = torch.rand(input_size, rules, len(params), requires_grad = True)

##1 Fuzzification Layer


###1.1 Proposal 1
To initialize it:
* data_shape: Shape of a batch of data
* rules : Number of initial rules (default = 1)
* mf: Function to be used as MemberShip Function. This function must be written in such a way that it receives a tensor as parameters (default = gaussian_3, Gaussian function with 3 parameters)
* params : List with the names of the MF parameters to use (default = ['sigma', 'mu', 'f'])
* init_mode: Premises initialization mode (default = 0)
  
The idea is that the layer stores the premises in a tensor that will have as an attribute, this implies that the mf that you want to apply must be written in a special way: 2 parameters, x (matrix of input data) and p (matrix of parameters to be used for each input data).

In [ ]:
def gaussian(x, mu, sigma, f=1):
    return torch.exp( -f * torch.pow((x - mu), 2) / (2*torch.pow(sigma, 2)) )

In [ ]:
def gaussian_3(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [ ]:
unic_tensor = torch.tensor([1, 2])
x_multi = torch.tensor([[1, 2],
                        [2, 4],
                        [3, 6]])

In [ ]:
def gaussian_2(x, p):
    return torch.exp(-torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [ ]:
params=['sigma', 'mu', 'f']
rules = 3
shape = x_multi.shape
shape

torch.Size([3, 2])

In [ ]:
x_multi.unsqueeze(2).repeat(1, 1, rules)

tensor([[[1, 1, 1],
         [2, 2, 2]],

        [[2, 2, 2],
         [4, 4, 4]],

        [[3, 3, 3],
         [6, 6, 6]]])

In [ ]:
premises = torch.rand(shape[1], rules, len(params))
#                    (       2,        3,           3)
premises

tensor([[[0.4360, 0.7640, 0.2019],
         [0.9393, 0.1089, 0.3274],
         [0.7702, 0.1134, 0.6209]],

        [[0.8206, 0.6474, 0.5873],
         [0.1844, 0.5747, 0.0944],
         [0.1423, 0.6080, 0.2762]]])

In [ ]:
y = gaussian_3(x_multi.unsqueeze(2).repeat(1, 1, rules), premises)
y

tensor([[[9.4647e-01, 9.5047e-01, 2.7942e-01],
         [3.7734e-01, 6.2430e-01, 2.7545e-01]],

        [[6.5503e-01, 1.8280e-07, 1.4003e-16],
         [8.3956e-04, 1.2484e-01, 3.8486e-03]],

        [[3.2075e-01, 3.6994e-26, 0.0000e+00],
         [6.8712e-09, 7.9582e-03, 2.7070e-06]]])

In [ ]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    Properties:
        data_shape : dimension of the input data
        rules : number of initial rules
        mf: function used as membership function
        params: list with parameter names of the mf
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, data_shape, rules=3, mf=gaussian_3, params=['sigma', 'mu', 'f'], init_mode=0):
        super(FuzzifyLayer, self).__init__()
        self.rules = rules
        self.data_shape = data_shape
        self.mf = mf
        self.params = params
        if init_mode == 0:
            self.premises = torch.rand(data_shape[1], rules, len(params))
        elif init_mode == 1:
            #ANOTHER WAY TO INITIALIZE THE PREMISES
            self.premises = torch.ones(data_shape[1], rules, len(params))

    def forward(self, x):
        return self.mf(x.unsqueeze(2).repeat(1, 1, self.rules), self.premises)

    def show_premises(self):
        print("Premises tensor:")
        print(self.premises)

    def premises(self):
        return self.premises

In [ ]:
train_data = torch.rand(5, 2)
train_data #5 2-dimensional input vectors (batch size = 5)

tensor([[0.6740, 0.1736],
        [0.1384, 0.2114],
        [0.8910, 0.0256],
        [0.7188, 0.1903],
        [0.7614, 0.2695]])

In [ ]:
fuzzify_layer = FuzzifyLayer(train_data.shape, rules=3)

In [ ]:
'''
x1 premises
[[[mu, sigma, f],   (feature 1)
  [mu, sigma, f],   (feature 2)
  [mu, sigma, f]],  (feature 3)

x2 premises
 [[mu, sigma, f],   (feature 1)
  [mu, sigma, f],   (feature 2)
  [mu, sigma, f]]]  (feature 3)
'''
fuzzify_layer.show_premises()

Premises tensor:
tensor([[[0.1801, 0.0451, 0.8473],
         [0.0221, 0.8151, 0.6243],
         [0.0838, 0.7763, 0.1876]],

        [[0.3382, 0.1925, 0.8397],
         [0.6066, 0.0174, 0.7629],
         [0.4391, 0.0212, 0.1589]]])


In [ ]:
output1 = fuzzify_layer(train_data)
'''
batch input 1
[[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]],  (x2)

batch input 2
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)

...
...

last batch input
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
output1

tensor([[[7.9030e-23, 8.1903e-01, 9.4724e-01],
         [7.3571e-01, 0.0000e+00, 4.1015e-06]],

        [[6.9515e-01, 9.9367e-01, 9.9954e-01],
         [8.3361e-01, 0.0000e+00, 1.0948e-04]],

        [[0.0000e+00, 7.0143e-01, 9.0359e-01],
         [3.3058e-01, 0.0000e+00, 8.6124e-14]],

        [[5.1529e-27, 7.9614e-01, 9.3919e-01],
         [7.8048e-01, 0.0000e+00, 1.8570e-05]],

        [[2.4249e-31, 7.7357e-01, 9.3105e-01],
         [9.4792e-01, 0.0000e+00, 6.3269e-03]]])

###1.1.O Proposal 1 Optimized and Flexible
The idea is to only use pytorch functions and allow operations for both an input data and a batch of data.

In [ ]:
def gaussian_3a(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x.unsqueeze(x.dim()) - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [ ]:
def gaussian_3b(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

#input is required to be x.unsqueeze(x.dim())

In [ ]:
uniq_tensor = torch.tensor([1, 2])
x_multi = torch.tensor([[1, 2],
                        [2, 4],
                        [3, 6]])

In [ ]:
premises = torch.tensor([[[10, 20, 1],
                          [30, 40, 2]],

                         [[50, 60, 3],
                          [70, 80, 4]]])

In [ ]:
gaussian_3a(uniq_tensor, premises)

tensor([[0.9037, 0.5912],
        [0.3829, 0.2357]])

In [ ]:
gaussian_3a(x_multi, premises)

tensor([[[0.9037, 0.5912],
         [0.3829, 0.2357]],

        [[0.9231, 0.6126],
         [0.4141, 0.2563]],

        [[0.9406, 0.6341],
         [0.4463, 0.2780]]])

De esta manera es posible definir diversas funciones para usarlas como MFs:

In [ ]:
uniq_tensor = torch.tensor([0.4681, 0.3757])
uniq_tensor

tensor([0.4681, 0.3757])

In [ ]:
x_multi = torch.tensor([[0.4681, 0.3757],
        [0.6850, 0.1338],
        [0.9057, 0.2981]])
x_multi

tensor([[0.4681, 0.3757],
        [0.6850, 0.1338],
        [0.9057, 0.2981]])

In [ ]:
#For functions with 4 parameters
premises4 = torch.rand(2, 2, 4)
premises4

tensor([[[0.4491, 0.6160, 0.1673, 0.7056],
         [0.6706, 0.9481, 0.7480, 0.8220]],

        [[0.5231, 0.9272, 0.9297, 0.8355],
         [0.2175, 0.1732, 0.2664, 0.3142]]])

In [ ]:
#For functions with 3 parameters
premises3 = torch.rand(2, 2, 3)
premises3

tensor([[[0.2515, 0.5746, 0.1477],
         [0.8040, 0.5384, 0.9120]],

        [[0.3446, 0.2671, 0.6166],
         [0.2655, 0.4770, 0.2416]]])

In [ ]:
#For functions with 2 parameters
premises2 = torch.rand(2, 2, 2)
premises2

tensor([[[0.0416, 0.5152],
         [0.7031, 0.9217]],

        [[0.8549, 0.7985],
         [0.5983, 0.3462]]])

In [ ]:
#INPUT x WILL BE CONSIDERED TO ALWAYS BE: x.unsqueeze(x.dim())

In [ ]:
gaussian2_params = ['sigma', 'mu']
def gaussian2(x, p):
    return torch.exp(torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [ ]:
gaussian2(uniq_tensor.unsqueeze(unic_tensor.dim()), premises2)

tensor([[1.4086, 1.0330],
        [1.1973, 1.2295]])

In [ ]:
gaussian2(x_multi.unsqueeze(x_multi.dim()), premises2)

tensor([[[1.4086, 1.0330],
         [1.1973, 1.2295]],

        [[2.1807, 1.0002],
         [1.5034, 2.4591]],

        [[4.0807, 1.0244],
         [1.2752, 1.4562]]])

In [ ]:
gaussian3_params = ['sigma', 'mu', 'f']
def gaussian3(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [ ]:
gaussian3(uniq_tensor.unsqueeze(unic_tensor.dim()), premises3)

tensor([[0.9896, 0.8374],
        [0.9958, 0.9936]])

In [ ]:
gaussian3(x_multi.unsqueeze(x_multi.dim()), premises3)

tensor([[[0.9896, 0.8374],
         [0.9958, 0.9936]],

        [[0.9588, 0.9780],
         [0.8252, 0.9908]],

        [[0.9087, 0.9839],
         [0.9907, 0.9994]]])

In [ ]:
bell_params = ["a", "b", "c"]
def bell(x, p):
    return 1 / (1 + torch.pow(torch.abs((x - p[:, :, 2]) / p[:, :, 0]), 2 * p[:, :, 1]))

In [ ]:
bell(uniq_tensor.unsqueeze(unic_tensor.dim()), premises3)

tensor([[0.4308, 0.6547],
        [0.5477, 0.6574]])

In [ ]:
bell(x_multi.unsqueeze(x_multi.dim()), premises3)

tensor([[[0.4308, 0.6547],
         [0.5477, 0.6574]],

        [[0.2947, 0.7961],
         [0.4551, 0.7026]],

        [[0.2196, 0.9946],
         [0.5105, 0.8141]]])

In [ ]:
triangular_params = ["a", "b"]
def triangular(x, p):
    return torch.clamp(1 - torch.abs((x - p[:, :, 1]) / p[:, :, 0]), min=0)

In [ ]:
triangular(uniq_tensor.unsqueeze(unic_tensor.dim()), premises2)

tensor([[0.0000, 0.3548],
        [0.5054, 0.9507]])

In [ ]:
triangular(x_multi.unsqueeze(x_multi.dim()), premises2)

tensor([[[0.0000, 0.3548],
         [0.5054, 0.9507]],

        [[0.0000, 0.6633],
         [0.2225, 0.6449]],

        [[0.0000, 0.9772],
         [0.4147, 0.9196]]])

In [ ]:
trapezoidal_params = ["a", "b", "c", "d"]
def trapezoidal(x, p):
    return torch.min(torch.clamp((x - p[:, :, 0]) / (p[:, :, 1] - p[:, :, 0]), min=0, max=1), torch.clamp((p[:, :, 3] - x) / (p[:, :, 3] - p[:, :, 2]), min=0, max=1))

In [ ]:
trapezoidal(uniq_tensor.unsqueeze(unic_tensor.dim()), premises4)

tensor([[0.1138, 0.0000],
        [0.0000, 0.0000]])

In [ ]:
trapezoidal(x_multi.unsqueeze(x_multi.dim()), premises4)

tensor([[[0.1138, 0.0000],
         [0.0000, 0.0000]],

        [[0.0382, 0.0520],
         [0.0000, 1.0000]],

        [[0.0000, 0.0000],
         [0.0000, 0.0000]]])

Ahora ya se tienen funciones que pueden usarse como MFs flexibles (permiten operaciones sobre un dato de entrada solo y también sobre batches de estos mismos)

In [ ]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    Attributes:
        input_size : size of a single input
        rules : number of initial rules
        mf: function used as membership function
        params: list with parameter names
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, rules=3, mf=gaussian3, params=['sigma', 'mu', 'f'], init_mode=0):
        super(FuzzifyLayer, self).__init__()
        self.input_size = input_size
        self.rules = rules
        self.mf = mf
        self.params = params
        if init_mode == 0:
            self.premises = torch.rand(input_size, rules, len(params))
        elif init_mode == 1:
            #ANOTHER WAY TO INITIALIZE THE PREMISES
            self.premises = torch.ones(input_size, rules, len(params))

    def forward(self, x):
        return self.mf(x.unsqueeze(x.dim()), self.premises)

    @property
    def premises_structure(self):
        print("Premises Structure:")
        for i in range(self.input_size):
            print(f"    x{i} parameters:")
            for j in range(self.rules):
                print(f"        feature {j + 1}:")
                [print(f"            {self.params[k]}: {self.premises[i, j, k]}") for k in range(len(self.params))]


In [ ]:
layer1_0 = FuzzifyLayer(x_multi.shape[1])

In [ ]:
print("Premises: ")
layer1_0.premises

Premises: 


tensor([[[0.9886, 0.1714, 0.6247],
         [0.6608, 0.5587, 0.4319],
         [0.3081, 0.1245, 0.6271]],

        [[0.6734, 0.3675, 0.5786],
         [0.9933, 0.6330, 0.3074],
         [0.5031, 0.7634, 0.1380]]])

In [ ]:
layer1_0.premises_structure

Premises Structure:
    x0 parameters:
        feature 1:
            sigma: 0.988573431968689
            mu: 0.1713951826095581
            f: 0.6246953010559082
        feature 2:
            sigma: 0.6607720851898193
            mu: 0.5586730241775513
            f: 0.43190181255340576
        feature 3:
            sigma: 0.30813390016555786
            mu: 0.12449318170547485
            f: 0.6270505785942078
    x1 parameters:
        feature 1:
            sigma: 0.6734197735786438
            mu: 0.3674948215484619
            f: 0.5785650014877319
        feature 2:
            sigma: 0.9932684898376465
            mu: 0.6329724192619324
            f: 0.30743563175201416
        feature 3:
            sigma: 0.5030530691146851
            mu: 0.7634319067001343
            f: 0.13800817728042603


In [ ]:
batch_data0 = torch.rand(5,2)
batch_data0

tensor([[0.4611, 0.6702],
        [0.7942, 0.2278],
        [0.4445, 0.0819],
        [0.3205, 0.7161],
        [0.5290, 0.8933]])

In [ ]:
single_data0 = torch.rand(2)
single_data0

tensor([0.2963, 0.0548])

In [ ]:
batch_output1_0 = layer1_0(batch_data0)
'''
batch input 1
[[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]],  (x2)

batch input 2
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)

...
...

last batch input
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
batch_output1_0

tensor([[[0.0519, 0.9728, 0.6230],
         [1.0000, 0.9608, 0.9967]],

        [[0.6691, 0.9878, 0.0084],
         [0.6535, 0.7987, 0.9911]],

        [[0.0430, 0.9682, 0.6865],
         [0.4726, 0.7271, 0.9792]],

        [[0.0087, 0.9230, 0.9969],
         [0.9961, 0.9710, 0.9946]],

        [[0.1059, 0.9881, 0.3726],
         [0.9016, 0.9962, 0.9821]]])

In [ ]:
single_output1_0 = layer1_0(single_data0)
'''
[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
 [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
single_output1_0

tensor([[0.0061, 0.9122, 0.9972],
        [0.4406, 0.7133, 0.9765]])

###1.1.2 Proposal 1 with initialization of MATLAB premises

In [ ]:
gaussian3_params = ['sigma', 'mu', 'f']
def gaussian3(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [ ]:
gaussian3_params = ['sigma', 'mu', 'f']
def gaussian3_matlab(x, p):
    return torch.exp(-p[:, :, 2] * (torch.pow((x - p[:, :, 0]) / p[:, :, 1], 2)))

In [ ]:
gaussian2_params = ['sigma', 'mu']
def gaussian2(x, p):
    return torch.exp(-(torch.pow((x - p[:, :, 0]) / p[:, :, 1], 2)))

In [ ]:
batch_data1 = torch.tensor([[0.6294, 0.2647, 0.9150, 0.9143, -0.1565],
   [0.8116, -0.8049, 0.9298, -0.0292, 0.8315],
   [-0.7460, -0.4430, -0.6848, 0.6006, 0.5844],
   [0.8268, 0.0938, 0.9412, -0.7162, 0.9190]])
batch_data1

tensor([[ 0.6294,  0.2647,  0.9150,  0.9143, -0.1565],
        [ 0.8116, -0.8049,  0.9298, -0.0292,  0.8315],
        [-0.7460, -0.4430, -0.6848,  0.6006,  0.5844],
        [ 0.8268,  0.0938,  0.9412, -0.7162,  0.9190]])

In [ ]:
mi = torch.min(batch_data1, dim=0).values
mi

tensor([-0.7460, -0.8049, -0.6848, -0.7162, -0.1565])

In [ ]:
ma = torch.max(batch_data1, dim=0).values
ma

tensor([0.8268, 0.2647, 0.9412, 0.9143, 0.9190])

In [ ]:
ex = ma - mi
ex

tensor([1.5728, 1.0696, 1.6260, 1.6305, 1.0755])

In [ ]:
rules = 3
dim = 5

In [ ]:
stp = ex / (rules - 1)
stp

tensor([0.7864, 0.5348, 0.8130, 0.8153, 0.5378])

In [ ]:
h1 = torch.arange(mi[0], ma[0] + stp[0], stp[0])
h1

tensor([-0.7460,  0.0404,  0.8268,  1.6132])

In [ ]:
h2 = torch.arange(mi[1], ma[1] + stp[1], stp[1])
h2

tensor([-0.8049, -0.2701,  0.2647])

In [ ]:
h3 = torch.arange(mi[2], ma[2] + stp[2], stp[2])
h3

tensor([-0.6848,  0.1282,  0.9412])

In [ ]:
h4 = torch.arange(mi[3], ma[3] + stp[3], stp[3])
h4

tensor([-0.7162,  0.0991,  0.9143])

In [ ]:
h5 = torch.arange(mi[4], ma[4] + stp[4], stp[4])
h5

tensor([-0.1565,  0.3813,  0.9190,  1.4568])

In [ ]:
pp = torch.rand(dim, rules, 3)
pp

tensor([[[0.2798, 0.5586, 0.1588],
         [0.5720, 0.5124, 0.3971],
         [0.7746, 0.5913, 0.3734]],

        [[0.8885, 0.0558, 0.0564],
         [0.4181, 0.8391, 0.1608],
         [0.1069, 0.3165, 0.8111]],

        [[0.9076, 0.4776, 0.6858],
         [0.5026, 0.0978, 0.2369],
         [0.5946, 0.7957, 0.8155]],

        [[0.4171, 0.3087, 0.4236],
         [0.6283, 0.4437, 0.2907],
         [0.8615, 0.0843, 0.4129]],

        [[0.1002, 0.1930, 0.1710],
         [0.3157, 0.4121, 0.1711],
         [0.6618, 0.4341, 0.0391]]])

In [ ]:
pp[0][1][2]

tensor(0.3971)

In [ ]:
pp = torch.zeros(dim, rules, 3)
for i in range(dim):
    h = torch.arange(mi[i], ma[i] + stp[i], stp[i])
    for j in range(rules):
        pp[i][j][0] = h[j]
        pp[i][j][1] = stp[i]/2
        pp[i][j][2] = 2
pp

tensor([[[-0.7460,  0.3932,  2.0000],
         [ 0.0404,  0.3932,  2.0000],
         [ 0.8268,  0.3932,  2.0000]],

        [[-0.8049,  0.2674,  2.0000],
         [-0.2701,  0.2674,  2.0000],
         [ 0.2647,  0.2674,  2.0000]],

        [[-0.6848,  0.4065,  2.0000],
         [ 0.1282,  0.4065,  2.0000],
         [ 0.9412,  0.4065,  2.0000]],

        [[-0.7162,  0.4076,  2.0000],
         [ 0.0991,  0.4076,  2.0000],
         [ 0.9143,  0.4076,  2.0000]],

        [[-0.1565,  0.2689,  2.0000],
         [ 0.3813,  0.2689,  2.0000],
         [ 0.9190,  0.2689,  2.0000]]])

In [ ]:
torch.std(batch_data1[:, 0])

tensor(0.7563)

In [ ]:
torch.mean(batch_data1[:, 0])

tensor(0.3805)

In [ ]:
pp0 = torch.zeros(dim, 1, 3)
for i in range(dim):
    for j in range(1):
        pp0[i][j][0] = torch.std(batch_data1[:, i])
        pp0[i][j][1] = torch.mean(batch_data1[:, i])
        pp0[i][j][2] = 2
pp0

tensor([[[ 0.7563,  0.3805,  2.0000]],

        [[ 0.4917, -0.2223,  2.0000]],

        [[ 0.8068,  0.5253,  2.0000]],

        [[ 0.7217,  0.1924,  2.0000]],

        [[ 0.4884,  0.5446,  2.0000]]])

In [ ]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    Attributes:
        input_size : size of a single input
        rules : number of initial rules
        mf: function used as membership function
        params: list with parameter names
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, rules=3, mf=gaussian3, params=['sigma', 'mu', 'f']):
        super(FuzzifyLayer, self).__init__()
        self.input_size = input_size
        self.rules = rules
        self.mf = mf
        self.params = params
        self.premises = torch.rand(input_size, rules, len(params))

    def forward(self, x):
        return self.mf(x.unsqueeze(x.dim()), self.premises)

    #For both gaussian_3 and gaussian_2 function
    def uniform_premises(self, x_train):
        if self.rules > 1:
            min = torch.min(x_train, dim=0).values
            max = torch.max(x_train, dim=0).values
            stp = (max - min) / (self.rules - 1)
            for i in range(self.input_size):
                h = torch.arange(mi[i], ma[i] + stp[i], stp[i])
                for j in range(self.rules):
                    self.premises[i][j][0] = h[j]
                    self.premises[i][j][1] = stp[i]/2
                    if (len(self.params) == 3): self.premises[i][j][2] = 2
        else:
            for i in range(self.input_size):
                for j in range(self.rules):
                    self.premises[i][j][0] = torch.std(x_train[:, i])
                    self.premises[i][j][1] = torch.mean(x_train[:, i])
                    if (len(self.params) == 3): self.premises[i][j][2] = 2

    @property
    def premises_structure(self):
        print("Premises Structure:")
        for i in range(self.input_size):
            print(f"    x{i} parameters:")
            for j in range(self.rules):
                print(f"        rule {j + 1}:")
                [print(f"            {self.params[k]}: {self.premises[i, j, k]}") for k in range(len(self.params))]

In [ ]:
layer1_matlab = FuzzifyLayer(batch_data1.shape[1])
layer1_matlab.uniform_premises(batch_data1)

In [ ]:
layer1_matlab.premises_structure

Premises Structure:
    x0 parameters:
        feature 1:
            sigma: -0.7459999918937683
            mu: 0.39319998025894165
            f: 2.0
        feature 2:
            sigma: 0.04039996862411499
            mu: 0.39319998025894165
            f: 2.0
        feature 3:
            sigma: 0.8267999291419983
            mu: 0.39319998025894165
            f: 2.0
    x1 parameters:
        feature 1:
            sigma: -0.8048999905586243
            mu: 0.26739999651908875
            f: 2.0
        feature 2:
            sigma: -0.2700999975204468
            mu: 0.26739999651908875
            f: 2.0
        feature 3:
            sigma: 0.2646999955177307
            mu: 0.26739999651908875
            f: 2.0
    x2 parameters:
        feature 1:
            sigma: -0.6848000288009644
            mu: 0.4065000116825104
            f: 2.0
        feature 2:
            sigma: 0.1281999945640564
            mu: 0.4065000116825104
            f: 2.0
        feature 3:
     

In [ ]:
output_1_matlab = layer1_matlab(batch_data1)

###1.P Some things to consider
* **COULD IT BE BETTER TO DEFINE THE GAUSSIAN FUNCTION WITHIN THE LAYER?** (I don't think so, the model would lose flexibility)
* **MATLAB PREMISES INITIALIZATION STILL MUST BE APPLIED**

###1.2 Proposal 2

In [ ]:
#STILL NOTHING

##2 Middle Layers

###2.1 Generalized "AND" Layer

In [ ]:
def multiplication_and(tensor):
    return tensor.transpose(1, 2).prod(dim=2)

In [ ]:
result = multiplication_and(output1)
result

tensor([[5.8143e-23, 0.0000e+00, 3.8851e-06],
        [5.7948e-01, 0.0000e+00, 1.0943e-04],
        [0.0000e+00, 0.0000e+00, 7.7820e-14],
        [4.0217e-27, 0.0000e+00, 1.7441e-05],
        [2.2987e-31, 0.0000e+00, 5.8907e-03]])

In [ ]:
class GeneralizedANDLayer(nn.Module):
    '''
    Class that represents the second layer of an ANFIS
    '''
    def __init__(self):
        super(GeneralizedANDLayer, self).__init__()

    def forward(self, x):
        return x.transpose(1, 2).prod(dim=2)

In [ ]:
class GeneralizedANDLayer2(nn.Module):
    '''
    Class that represents the second layer of an ANFIS

    Propertues:
        and_function: function that is used as an AND operator
    '''
    def __init__(self, and_function=multiplication_and):
        super(GeneralizedANDLayer2, self).__init__()
        self.and_function = and_function

    def forward(self, x):
        return self.and_function(x)

In [ ]:
generalizedAND_layer = GeneralizedANDLayer2()

In [ ]:
output2 = generalizedAND_layer(output1)
'''
[[w1, w2, w3],  (of batch input1)
 [w1, w2, w3],  (of batch input2)
 ...
 [w1, w2, w3]]  (of last batch input)
'''
output2

tensor([[5.8143e-23, 0.0000e+00, 3.8851e-06],
        [5.7948e-01, 0.0000e+00, 1.0943e-04],
        [0.0000e+00, 0.0000e+00, 7.7820e-14],
        [4.0217e-27, 0.0000e+00, 1.7441e-05],
        [2.2987e-31, 0.0000e+00, 5.8907e-03]])

###2.2 Normalization Layer


In [ ]:
tensor_input = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
normalized_tensor = torch.nn.functional.normalize(tensor_input)
print(normalized_tensor)

tensor([[0.2673, 0.5345, 0.8018],
        [0.4558, 0.5698, 0.6838]])


In [ ]:
class NormalizationLayer(nn.Module):
    '''
    Class that represents the third layer of an ANFIS
    '''
    def __init__(self):
        super(NormalizationLayer, self).__init__()

    def forward(self, x):
        return torch.nn.functional.normalize(x)

In [ ]:
normalization_layer = NormalizationLayer()

In [ ]:
output3 = normalization_layer(output2)
''' ~w = normalization(w)
[[~w1, ~w2, ~w3],  (of batch input1)
 [~w1, ~w2, ~w3],  (of batch input2)
 ...
 [~w1, ~w2, ~w3]]  (of last batch input)
'''
output3

tensor([[1.4966e-17, 0.0000e+00, 1.0000e+00],
        [1.0000e+00, 0.0000e+00, 1.8884e-04],
        [0.0000e+00, 0.0000e+00, 7.7820e-02],
        [2.3059e-22, 0.0000e+00, 1.0000e+00],
        [3.9022e-29, 0.0000e+00, 1.0000e+00]])

###2.3 Generalized "AND" + Normalization Layer (Normalized Firing levels layer)
The idea is to do the AND and normalization operation in a single layer (since there are no trainable parameters here), in this way the outputs would be the already normalized firing levels

In [ ]:
def multiplication_and(tensor):
    return tensor.transpose(1, 2).prod(dim=2)

In [ ]:
class FiringLevelsLayer(nn.Module):
    '''
    Class that represents the second layer of an ANFIS

    Attributes:
        and_function: function that is used as an AND operator
    '''
    def __init__(self, and_function=multiplication_and):
        super(FiringLevelsLayer, self).__init__()
        self.and_function = and_function

    def forward(self, x):
        return torch.nn.functional.normalize(self.and_function(x))

In [ ]:
firing_levels_layer = FiringLevelsLayer()

In [ ]:
output23 = firing_levels_layer(output1)
''' ~w = normalization(w)
[[~w1, ~w2, ~w3],  (of batch input1)
 [~w1, ~w2, ~w3],  (of batch input2)
 ...
 [~w1, ~w2, ~w3]]  (of last batch input)
'''
output23

tensor([[1.4966e-17, 0.0000e+00, 1.0000e+00],
        [1.0000e+00, 0.0000e+00, 1.8884e-04],
        [0.0000e+00, 0.0000e+00, 7.7820e-02],
        [2.3059e-22, 0.0000e+00, 1.0000e+00],
        [3.9022e-29, 0.0000e+00, 1.0000e+00]])

###2.3.O Normalized Firing Levels Layer Optimized and Flexible
The idea is to only use pytorch functions and allow operations for both an input data and a batch of data.

In [ ]:
single_output1_0 = layer1_0(single_data0)
'''
[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
 [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
single_output1_0

tensor([[0.0061, 0.9122, 0.9972],
        [0.4406, 0.7133, 0.9765]])

In [ ]:
batch_output1_0 = layer1_0(batch_data0)
'''
batch input1
[[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]],  (x2)

batch input2
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)

...
...

last batch input
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
batch_output1_0

tensor([[[0.0519, 0.9728, 0.6230],
         [1.0000, 0.9608, 0.9967]],

        [[0.6691, 0.9878, 0.0084],
         [0.6535, 0.7987, 0.9911]],

        [[0.0430, 0.9682, 0.6865],
         [0.4726, 0.7271, 0.9792]],

        [[0.0087, 0.9230, 0.9969],
         [0.9961, 0.9710, 0.9946]],

        [[0.1059, 0.9881, 0.3726],
         [0.9016, 0.9962, 0.9821]]])

In [ ]:
batch_output1_0.dim()

3

In [ ]:
single_output1_0.dim()

2

In [ ]:
def multiplication_and(x):
    return x.prod(dim=x.dim()-2)

In [ ]:
batch_firing_levels = multiplication_and(batch_output1_0)
''' wi = feature [i] firing level
[[w1, w2, w3],  (of batch input1)
 [w1, w2, w3],  (of batch input2)
 ...
 [w1, w2, w3]]  (of last batch input)
'''
batch_firing_levels

tensor([[0.0519, 0.9346, 0.6209],
        [0.4373, 0.7889, 0.0083],
        [0.0203, 0.7039, 0.6722],
        [0.0087, 0.8962, 0.9916],
        [0.0955, 0.9843, 0.3659]])

In [ ]:
batch_firing_levels.dim()

2

In [ ]:
torch.nn.functional.normalize(batch_firing_levels, p=1, dim=-1)

tensor([[0.0323, 0.5814, 0.3863],
        [0.3542, 0.6390, 0.0067],
        [0.0145, 0.5041, 0.4814],
        [0.0046, 0.4726, 0.5229],
        [0.0660, 0.6808, 0.2531]])

In [ ]:
torch.nn.functional.normalize(batch_firing_levels, p=1, dim=1)

tensor([[0.0323, 0.5814, 0.3863],
        [0.3542, 0.6390, 0.0067],
        [0.0145, 0.5041, 0.4814],
        [0.0046, 0.4726, 0.5229],
        [0.0660, 0.6808, 0.2531]])

In [ ]:
single_firing_levels = multiplication_and(single_output1_0)
''' wi = feature [i] firing level
[w1, w2, w3]
'''
single_firing_levels

tensor([0.0027, 0.6506, 0.9737])

In [ ]:
#normalization should result: [0.3792, 0.2889, 0.3318]

In [ ]:
torch.nn.functional.normalize(single_firing_levels, p=1, dim=0)

tensor([0.0017, 0.3999, 0.5985])

In [ ]:
torch.nn.functional.normalize(single_firing_levels, p=1, dim=-1)

tensor([0.0017, 0.3999, 0.5985])

In [ ]:
class FiringLevelsLayer(nn.Module):
    '''
    Class used to calculate firing levels and normalize them
    '''
    def __init__(self):
        super(FiringLevelsLayer, self).__init__()

    def forward(self, x):
        return torch.nn.functional.normalize(x.prod(dim=x.dim()-2), p=1, dim=-1)

In [ ]:
layer23_0 = FiringLevelsLayer()

In [ ]:
single_normalized_FL = layer23_0(single_output1_0)
''' ~wi = normalization(wi)
    wi = feature [i] firing level
[~w1, ~w2, ~w3]
'''
single_normalized_FL

tensor([0.0017, 0.3999, 0.5985])

In [ ]:
batch_normalized_FL = layer23_0(batch_output1_0)
''' ~w = normalization(w)
    wi = feature [i] firing level
[[~w1, ~w2, ~w3],  (batch input1)
 [~w1, ~w2, ~w3],  (batch input2)
 ...
 [~w1, ~w2, ~w3]]  (last batch input)
'''
batch_normalized_FL

tensor([[0.0323, 0.5814, 0.3863],
        [0.3542, 0.6390, 0.0067],
        [0.0145, 0.5041, 0.4814],
        [0.0046, 0.4726, 0.5229],
        [0.0660, 0.6808, 0.2531]])

###Testeo de pesos ponderados matlab



In [ ]:
layer2_matlab = FiringLevelsLayer()

In [ ]:
output_2_matlab = layer2_matlab(output_1_matlab)
output_2_matlab

tensor([[1.1257e-19, 1.5007e-01, 8.4993e-01],
        [3.9147e-15, 9.9892e-01, 1.0829e-03],
        [8.6307e-05, 9.9991e-01, 4.9215e-14],
        [1.9157e-19, 1.9100e-01, 8.0900e-01]])

###2.P Some things to consider
* **IT SHOULD BE MORE OPTIMAL TO PERFORM THE NORMALIZATION IN THE CONSEQUENT PART, that is, when the w is multiplied with the consequent function** (apparently it will be the same, however it is likely that using pytorch's functional.normalize is better )
* **WOULD IT BE USEFUL TO EXTRACT/SHOW THE FIRING LEVELS ON SCREEN WITHOUT NORMALIZING??**(should be possible in the anfis class)
* ** MUST VERIFY THAT THE NORMALIZATION USED HERE IS THE SAME ONE APPLIED IN MATLAB, the one in this jupyter notebook is a simple normalization of the type: [1, 2] -> [1/(1+2), 2/ (1+2)], the use of any exponent or special function in matlab has not been verified**

##3 Consequent Layer

###3.1 Proposal 1

In [ ]:
x = torch.tensor([[1, 2],
                  [3, 4]], dtype=torch.float32)
'''
[[x1, x2],  (batch input1)
 [x1, x2]]  (batch input2)
'''

c = torch.tensor([[10, 20, 30],
                  [40, 50, 60]], dtype=torch.float32)
'''
[[p1, q1, r1],  (feature 1 consequent parameters)
 [p2, q2, r2]]  (feature 2 consequent parameters)
'''

w = torch.tensor([[100, 200],
                  [300, 400]], dtype=torch.float32)
'''
[[~w1, ~w2],  (batch input 1 normalized firing levels)
 [~w1, ~w2]]  (batch input 2 normalized firing levels)
'''

'\n[[~w1, ~w2],  (batch input 1 normalized firing levels)\n [~w1, ~w2]]  (batch input 2 normalized firing levels)\n'

In [ ]:
def simple_linear(x, c):
    return x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1].unsqueeze(0)

In [ ]:
'''
[[x1*p1 + x2*q1 + r1, x1*p2 + x2*q2 + r2],  (x1, x2 of batch input1)
 [x1*p1 + x2*q1 + r1, x1*p2 + x2*q2 + r2]]  (x1, x2 of batch input2)
'''
simple_linear(x, c)

tensor([[ 80., 200.],
        [140., 380.]])

In [ ]:
def typical_linear(x, c, w):
    return (x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1].unsqueeze(0)).mul(w)

In [ ]:
'''
[[~w1*(x1*p1 + x2*q1 + r1), ~w2*(x1*p2 + x2*q2 + r2)],  (x1, x2, ~w1 y ~w2 of batch input1)
 [~w1*(x1*p1 + x2*q1 + r1), ~w2*(x1*p2 + x2*q2 + r2)]]  (x1, x2, ~w1 y ~w2 of batch input2)
'''
typical_linear(x, c, w)

tensor([[  8000.,  40000.],
        [ 42000., 152000.]])

In [ ]:
class ConsequentLayer(nn.Module):
    '''
    Class that represents the fourth layer (consequent layer) of an ANFIS

    Attributes:
        data_shape : dimension of the input data
        rules : number of rules
        function : function used as a consequent function
        consequents: array with the parameters used in each node of this layer
    '''
    def __init__(self, data_shape, rules=3, function=typical_linear):
        super(ConsequentLayer, self).__init__()
        self.rules = rules
        self.data_shape = data_shape
        self.function = function
        #CONSEQUENT PARAMETERS inicializados aleatoriamente de momento
        self.consequents = torch.rand(rules, data_shape[1] + 1)

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    def show_consequents(self):
        print("Consequents tensor:")
        print(self.consequents)

    def consequents(self):
        return self.consequents

In [ ]:
consequent_layer = ConsequentLayer(train_data.shape)

In [ ]:
# x : input data (for train)
train_data

tensor([[0.6740, 0.1736],
        [0.1384, 0.2114],
        [0.8910, 0.0256],
        [0.7188, 0.1903],
        [0.7614, 0.2695]])

In [ ]:
# c : consequent parameters by feature
consequent_layer.consequents

tensor([[0.6079, 0.4989, 0.6388],
        [0.8717, 0.7026, 0.5970],
        [0.4174, 0.8788, 0.4393]])

In [ ]:
# w: normalized firing levels for input on batch
output23

tensor([[1.4966e-17, 0.0000e+00, 1.0000e+00],
        [1.0000e+00, 0.0000e+00, 1.8884e-04],
        [0.0000e+00, 0.0000e+00, 7.7820e-02],
        [2.3059e-22, 0.0000e+00, 1.0000e+00],
        [3.9022e-29, 0.0000e+00, 1.0000e+00]])

In [ ]:
output4 = consequent_layer(train_data, output23)
''' ~w = normalization(w)   --> normalized firing levels
    fi = x1*pi + x2*qi + ri --> consequent function of feature i applied to an input (x1, x2)

[[~w1*f1, ~w2*f2, ~w3*f3],  (of batch input1)
 [~w1*f1, ~w2*f2, ~w3*f3],   (of batch input2)
 ...
 [~w1*f1, ~w2*f2, ~w3*f3]]   (of batch last input)
'''
output4

tensor([[1.6988e-17, 0.0000e+00, 8.7321e-01],
        [8.2839e-01, 0.0000e+00, 1.2896e-04],
        [0.0000e+00, 0.0000e+00, 6.4883e-02],
        [2.6994e-22, 0.0000e+00, 9.0654e-01],
        [4.8234e-29, 0.0000e+00, 9.9393e-01]])

###3.1.O Proposal 1 Optimized and Flexible

In [ ]:
single_normalized_FL

tensor([0.0017, 0.3999, 0.5985])

In [ ]:
batch_normalized_FL

tensor([[0.0323, 0.5814, 0.3863],
        [0.3542, 0.6390, 0.0067],
        [0.0145, 0.5041, 0.4814],
        [0.0046, 0.4726, 0.5229],
        [0.0660, 0.6808, 0.2531]])

In [ ]:
single_data0

tensor([0.2963, 0.0548])

In [ ]:
batch_data0

tensor([[0.4611, 0.6702],
        [0.7942, 0.2278],
        [0.4445, 0.0819],
        [0.3205, 0.7161],
        [0.5290, 0.8933]])

In [ ]:
params = ["p", "q", "r"]
#the number of parameters depends on the dimension of the input data

consequents = torch.rand(layer1_0.rules, len(params))
'''
[[p1, q1, r1],  (consequent parameters of feature 1)
 [p2, q2, r2]]  (consequent parameters of feature 2)
 [p3, q3, r3]]  (consequent parameters of feature 3)
'''
consequents

tensor([[0.7910, 0.1506, 0.5670],
        [0.3077, 0.2610, 0.2580],
        [0.8053, 0.2552, 0.8589]])

In [ ]:
def simple_linear(x, c):
    return x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1].unsqueeze(0)

In [ ]:
'''fi(x1, x2) = x1*pi + x2*qi + ri, con pi, qi, ri consequent parameters of feature i

[[f1(x1, x2), f2(x1, x2), f3(x1, x2)],  ((x1, x2) of batch input1)
 [f1(x1, x2), f2(x1, x2), f3(x1, x2)]]  ((x1, x2) of batch input2)
'''
simple_linear(batch_data0, consequents)

tensor([[1.0327, 0.5748, 1.4012],
        [1.2296, 0.5618, 1.5566],
        [0.9310, 0.4162, 1.2377],
        [0.9284, 0.5435, 1.2998],
        [1.1200, 0.6539, 1.5129]])

In [ ]:
'''fi(x1, x2) = x1*pi + x2*qi + ri, con pi, qi, ri consequent parameters of feature i

[f1(x1, x2), f2(x1, x2), f3(x1, x2)]  ((x1, x2) the input values)
'''
simple_linear(single_data0, consequents)

tensor([[0.8096, 0.3635, 1.1115]])

In [ ]:
def weighted_linear(x, c, w):
    return (x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1]).mul(w)

In [ ]:
weighted_linear(single_data0, consequents, single_normalized_FL)

tensor([0.0013, 0.1453, 0.6652])

In [ ]:
weighted_linear(batch_data0, consequents, batch_normalized_FL)

tensor([[0.0333, 0.3342, 0.5413],
        [0.4355, 0.3590, 0.0105],
        [0.0135, 0.2098, 0.5958],
        [0.0042, 0.2569, 0.6796],
        [0.0740, 0.4452, 0.3829]])

In [ ]:
single_data0.shape

torch.Size([2])

In [ ]:
single_data0.shape[-1]

2

In [ ]:
batch_data0.shape

torch.Size([5, 2])

In [ ]:
single_data0.shape[-1]

2

In [ ]:
class ConsequentLayer(nn.Module):
    '''
    Class that represents the fourth layer (consequent layer) of an ANFIS

    Attributes:
        input_size : size of a single input
        rules : number of rules
        function : function used as a consequent function
        consequents: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, rules=3, function=weighted_linear):
        super(ConsequentLayer, self).__init__()
        self.rules = rules
        self.input_size = input_size
        self.function = function
        #CONSEQUENT PARAMETERS randomly initialized at the moment
        self.consequents = torch.rand(rules, input_size + 1)

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    @property
    def consequents_structure(self):
        print("Consequents Structure:")
        for i in range(self.rules):
            print(f"    rule {i + 1} consequent parameters: {self.consequents[i]}")

In [ ]:
layer4_0 = ConsequentLayer(single_data0.shape[0])

In [ ]:
single_data0

tensor([0.2963, 0.0548])

In [ ]:
single_normalized_FL

tensor([0.0017, 0.3999, 0.5985])

In [ ]:
layer4_0.consequents_structure

Consequents Structure:
    feature 1 consequent parameters: tensor([0.3620, 0.5993, 0.8913])
    feature 2 consequent parameters: tensor([0.9173, 0.6461, 0.6236])
    feature 3 consequent parameters: tensor([0.5883, 0.6614, 0.9515])


In [ ]:
single_ponderated_consequents = layer4_0(single_data0, single_normalized_FL)
single_ponderated_consequents

tensor([0.0017, 0.3722, 0.6954])

In [ ]:
batch_ponderated_consequents = layer4_0(batch_data0, batch_normalized_FL)
batch_ponderated_consequents

tensor([[0.0471, 0.8603, 0.6435],
        [0.4659, 0.9581, 0.0106],
        [0.0160, 0.5465, 0.6099],
        [0.0066, 0.6523, 0.8437],
        [0.1069, 1.1479, 0.4692]])

###Testeo de salidas matlab

In [ ]:
pc = torch.tensor([[0.6557, 0.9340, 0.7431, 0.1712, 0.2769, 0.8235],
    [0.0357, 0.6787, 0.3922, 0.7060, 0.0462, 0.6948],
    [0.8491, 0.7577, 0.6555, 0.0318, 0.0971, 0.3171]])

In [ ]:
class TestingConsequentLayer(nn.Module):
    def __init__(self, input_size, rules=3, function=weighted_linear, pc=pc):
        super(TestingConsequentLayer, self).__init__()
        self.rules = rules
        self.input_size = input_size
        self.function = function
        self.consequents = pc

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    @property
    def consequents_structure(self):
        print("Consequents Structure:")
        for i in range(self.rules):
            print(f"    rule {i + 1} consequent parameters: {self.consequents[i]}")

In [ ]:
layer3_matlab = TestingConsequentLayer(batch_data1.shape[1])

In [ ]:
output_3_matlab = layer3_matlab(batch_data1, output_2_matlab)
output_3_matlab

tensor([[ 2.5627e-19,  2.8424e-01,  1.4158e+00],
        [ 5.9506e-15,  5.5935e-01,  1.1756e-03],
        [-2.7933e-05,  5.4990e-01, -5.0447e-14],
        [ 4.3765e-19,  1.3254e-01,  1.4349e+00]])

##4 Output Layer

###4.1 Proposal 1

In [ ]:
def sum(x):
    return torch.sum(x, dim=-1)

In [ ]:
sum(single_ponderated_consequents)

tensor(1.0693)

In [ ]:
sum(batch_ponderated_consequents)

tensor([1.5509, 1.4346, 1.1725, 1.5025, 1.7239])

In [ ]:
class OutputLayer(nn.Module):
    '''
    Class that represents the fifth layer (output layer) of an ANFIS
    '''
    def __init__(self, function=sum):
        super(OutputLayer, self).__init__()
        self.function = function

    def forward(self, x):
        return self.function(x)

###Testeo output matlab

In [ ]:
layer4_matlab = OutputLayer()

In [ ]:
output_4_matlab = layer4_matlab(output_3_matlab)
output_4_matlab

tensor([1.7000, 0.5605, 0.5499, 1.5674])

##5 ANFIS Structure Implementation corrections


###5.1 Proposal 1

In [ ]:
#Data set
train_data = torch.rand(10,3)
x_train = train_data[:, :-1]
y_train = train_data[:, -1]

In [ ]:
x_train

tensor([[0.9932, 0.5734],
        [0.2708, 0.0162],
        [0.5242, 0.2169],
        [0.0669, 0.8555],
        [0.7338, 0.1281],
        [0.9464, 0.6705],
        [0.0831, 0.2608],
        [0.5102, 0.7339],
        [0.7999, 0.7355],
        [0.2474, 0.8847]])

In [ ]:
class Type3ANFIS(nn.Module):
    '''
    Class that represents a type3 ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of initial rules
        cf: function used as consequent function
        mf: function used as membership function
        of: function used to join the outputs of each of the rules
        mf_params: list with MF parameters names
        premises: array with the parameters used in each node of this layer
        prem_init_mode: way to initialize the premises
            =0 : to initialize them randomly
            =1 : to initialize them [MATLAB IMPLEMENTATION MUST BE IMPLEMENTED]
    '''
    def __init__(self, input_size, init_rules=3, cf=weighted_linear, mf=gaussian3, of=sum, mf_params=['sigma', 'mu', 'f'], prem_init_mode=0):
        super(Type3ANFIS, self).__init__()
        self.fuzzify_layer = FuzzifyLayer(input_size, init_rules, mf, mf_params)
        self.firing_levels_layer = FiringLevelsLayer()
        self.consequent_layer = ConsequentLayer(input_size, init_rules, cf)
        self.output_layer = OutputLayer(of)

    def forward(self, x):
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.firing_levels_layer(output))
        output = self.output_layer(output)
        return output

In [ ]:
anfis1 = Type3ANFIS(x_train.shape[1])

In [ ]:
anfis1(x_train[0])

tensor(1.5665)

###5.2 Proposal 2

In [ ]:
class Type3ANFIS(nn.Module):
    '''
    Class that represents a type3 ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of initial rules
        cf: function used as consequent function
        mf: function used as membership function
        of: function used to join the outputs of each of the rules
        mf_params: list with MF parameters namesr
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly
    '''
    def __init__(self, input_size, init_rules=3, cf=weighted_linear, mf=gaussian3, of=sum, mf_params=['sigma', 'mu', 'f'], prem_init_mode=0, x_train=[]):
        super(Type3ANFIS, self).__init__()
        self.fuzzify_layer = FuzzifyLayer(input_size, init_rules, mf, mf_params)
        if x_train != []:
            fuzzify_layer.uniform_premises(x_train)
        self.firing_levels_layer = FiringLevelsLayer()
        self.consequent_layer = ConsequentLayer(input_size, init_rules, cf)
        self.output_layer = OutputLayer(of)

    def forward(self, x):
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.firing_levels_layer(output))
        output = self.output_layer(output)
        return output

In [ ]:
#Data set
train_data = torch.rand(10,3)
x_train = train_data[:, :-1]
y_train = train_data[:, -1]

In [ ]:
test = Type3ANFIS(x_train.shape[1])
test_batch_output = test(x_train)
test_batch_output

tensor([1.0434, 1.5950, 1.2757, 1.2972, 1.4296, 1.0419, 1.1285, 0.7691, 0.9735,
        0.5550])

In [ ]:
test_single_output = test(x_train[2])
test_single_output

tensor(1.2757)

##6 Finals for now

In [ ]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    To initialize it:
        input_size : size of a single input
        rules : number of initial rules
        mf: function used as membership function
        params: list with parameter names
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly

    Other attributes:
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, rules=3, mf=gaussian3, params=['sigma', 'mu', 'f'], x_train=[]):
        super(FuzzifyLayer, self).__init__()
        self.input_size = input_size
        self.rules = rules
        self.mf = mf
        self.params = params
        if x_train == []:
            self.premises = Parameter(torch.rand(input_size, rules, len(params)), requires_grad=True)
        else:
            self.premises = Parameter(torch.rand(input_size, rules, len(params)), requires_grad=False)
            if self.rules > 1:
                min = torch.min(x_train, dim=0).values
                max = torch.max(x_train, dim=0).values
                stp = (max - min) / (self.rules - 1)
                for i in range(self.input_size):
                    h = torch.arange(min[i], max[i] + stp[i], stp[i])
                    for j in range(self.rules):
                        self.premises[i][j][0] = h[j]
                        self.premises[i][j][1] = stp[i]/2
                        if (len(self.params) == 3): self.premises[i][j][2] = 2
            else:
                for i in range(self.input_size):
                    for j in range(self.rules):
                        self.premises[i][j][0] = torch.std(x_train[:, i])
                        self.premises[i][j][1] = torch.mean(x_train[:, i])
                        if (len(self.params) == 3): self.premises[i][j][2] = 2
            self.premises.requires_grad = True

    def forward(self, x):
        return self.mf(x.unsqueeze(x.dim()), self.premises)

    @property
    def premises_structure(self):
        print("Premises Structure:")
        for i in range(self.input_size):
            print(f"    x{i} parameters:")
            for j in range(self.rules):
                print(f"        rule {j + 1}:")
                [print(f"            {self.params[k]}: {self.premises[i, j, k]}") for k in range(len(self.params))]

In [ ]:
class FiringLevelsLayer(nn.Module):
    '''
    Class used to calculate firing levels and normalize them
    '''
    def __init__(self):
        super(FiringLevelsLayer, self).__init__()

    def forward(self, x):
        return torch.nn.functional.normalize(x.prod(dim=x.dim()-2), p=1, dim=-1)

In [ ]:
class ConsequentLayer(nn.Module):
    '''
    Class that represents the fourth layer (consequent layer) of an ANFIS

    To initialize it:
        input_size : size of a single input
        rules : number of rules
        function : function used as a consequent function

    Other attributes:
        consequents: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, rules=3, function=weighted_linear):
        super(ConsequentLayer, self).__init__()
        self.rules = rules
        self.input_size = input_size
        self.function = function
        #CONSEQUENT PARAMETERS randomly initialized at the moment
        self.consequents = Parameter(torch.rand(rules, input_size + 1), requires_grad=True)

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    @property
    def consequents_structure(self):
        print("Consequents Structure:")
        for i in range(self.rules):
            print(f"    rule {i + 1} consequent parameters: {self.consequents[i]}")

In [ ]:
class OutputLayer(nn.Module):
    '''
    Class that represents the fifth layer (output layer) of an ANFIS
    '''
    def __init__(self, function=sum):
        super(OutputLayer, self).__init__()
        self.function = function

    def forward(self, x):
        return self.function(x)

In [ ]:
class Type3ANFIS(nn.Module):
    '''
    Class that represents a type3 ANFIS

    To initialize it:
        input_size : size of a single input
        rules : number of initial rules
        cf: function used as consequent function
        mf: function used as membership function
        of: function used to join the outputs of each of the rules
        mf_params: list with MF parameters namesr
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly
    '''
    def __init__(self, input_size, rules=3, cf=weighted_linear, mf=gaussian3, of=sum, mf_params=['sigma', 'mu', 'f'], x_train=[]):
        super(Type3ANFIS, self).__init__()
        if x_train == []:
            self.fuzzify_layer = FuzzifyLayer(input_size, rules, mf, mf_params)
        else:
            self.fuzzify_layer = FuzzifyLayer(input_size, rules, mf, mf_params, x_train)
        self.firing_levels_layer = FiringLevelsLayer()
        self.consequent_layer = ConsequentLayer(input_size, rules, cf)
        self.output_layer = OutputLayer(of)

    def forward(self, x):
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.firing_levels_layer(output))
        output = self.output_layer(output)
        return output

    @property
    def premises_structure(self):
        self.fuzzify_layer.premises_structure

    @property
    def premises(self):
        return self.fuzzify_layer.premises

    @property
    def consequents_structure(self):
        self.consequent_layer.consequents_structure

    @property
    def consequents(self):
        return self.consequent_layer.consequents